# Intake phase pipeline: minimal version

Load epochs → phase at t=0 (non-causal and causal) → compare with BOSS. All logic is in `functions.py`; see `phase_pipeline_walkthrough.ipynb` for the full explanation.

In [ ]:
import os, sys
from pathlib import Path
import matplotlib.pyplot as plt

NEW_CODE = Path.cwd() if (Path.cwd() / "functions.py").exists() else Path.cwd() / "new_code"
sys.path[:0] = [str(NEW_CODE), str(NEW_CODE.parent)]
import functions as f
import settings as s   # all parameters (paths, band, causal params) live in settings.py

DEMO = False or os.environ.get("PHASE_NOTEBOOK_DEMO") == "1"   # True = synthetic epochs, no data needed
SUBJECTS = ["sub_102", "sub_103", "sub_104"]
TOLERANCE_DEG = 30

## 1. Load: `X` is (n_epochs, n_times)

In [ ]:
data = {sub: f.make_synthetic_epochs(subject=sub, seed=i) if DEMO else f.get_data(sub, s.LOAD_CONFIG)
        for i, sub in enumerate(SUBJECTS)}
d = data[SUBJECTS[0]]
d.X.shape, d.fs, d.labels[:5]

## 2. One epoch: non-causal (offline) vs causal (online) phase

In [ ]:
x = d.X[0]
nc = f.noncausal_phase(x, d.times_ms, d.fs, s.BAND, s.FILTER_ORDER, s.CUTOFF_MS)
c = f.causal_phase(x, d.times_ms, d.fs, s.BAND, s.CUTOFF_MS, s.MANUAL_CAUSAL_PARAMS)
print(f"BOSS {d.labels[0]} | non-causal {nc['phase_deg']:.0f} deg | causal {c.phase_deg:.0f} deg")

## 3. All epochs, all subjects

In [ ]:
estimates = f.estimate_all_subjects(
    data.values(), s.BAND, s.FILTER_ORDER, s.CUTOFF_MS, n_trials=False,
    causal_estimation=True, causal_params_mode="manual",
    manual_causal_params=s.MANUAL_CAUSAL_PARAMS, optimization_config=s.CAUSAL_OPTIMIZATION_CONFIG,
)
analysis = f.analyze_phase_results({sub: f.estimates_to_arrays(e) for sub, e in estimates.items()}, s.TOLERANCES_DEG)
{sub: round(a["counts"]["noncausal"][str(TOLERANCE_DEG)]["success_pct"], 1) for sub, a in analysis["subjects"].items()}

## 4. Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6), subplot_kw={"projection": "polar"})
f.plot_deviation_histogram(analysis["pooled"]["boss_target"], TOLERANCE_DEG, title="Phase - BOSS target", ax=axes[0])
f.plot_deviation_histogram(analysis["pooled"]["causal_error"], TOLERANCE_DEG, title="Causal - non-causal", ax=axes[1])
f.plot_success_vs_tolerance(analysis["success"])
plt.show()